In [41]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from scipy.stats import chi2_contingency, ttest_ind
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin

In [20]:
def rows_selection(data):
    return data[~((data["years_at_company"] == 0) & (data.age > 45))]

def Chi_squared_test(data):
    df = rows_selection(data)
    # Select categorical columns
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns

    # Define target column
    target = 'attrition'

    # Ensure target exists
    if target not in df.columns:
        raise ValueError(f"'{target}' column not found in DataFrame.")

    # Store results
    results = []

    for col in categorical_cols:
        if col != target:
            contingency_table = pd.crosstab(df[col], df[target])
            chi2, p, dof, expected = chi2_contingency(contingency_table)
            significance = "✅" if p < 0.05 else "❌"
            results.append({
                'Feature': col,
                'Chi2': round(chi2, 3),
                'p-value': round(p, 5),
                'DOF': dof,
                'Significant': significance
            })

    # Convert to DataFrame
    chi_square_df = pd.DataFrame(results).sort_values(by='p-value')
    selected_features_chi_square = chi_square_df[chi_square_df['p-value'] < 0.05]['Feature'].tolist()
    return selected_features_chi_square

def t_test_feature_selection(data):
    df = rows_selection(data)
    
    target = 'attrition'

    # Ensure target exists
    if target not in df.columns:
        raise ValueError(f"'{target}' column not found in DataFrame.")

    # Select numeric columns
    numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns

    # Ensure binary target (like Left/Stayed)
    groups = df[target].unique()
    if len(groups) != 2:
        raise ValueError(f"T-test requires a binary target variable, found: {groups}")

    # Store results
    results = []

    for col in numeric_cols:
        group1 = df[df[target] == groups[0]][col].dropna()
        group2 = df[df[target] == groups[1]][col].dropna()
        
        t_stat, p_val = ttest_ind(group1, group2, equal_var=False)  # Welch’s t-test
        significance = "✅" if p_val < 0.05 else "❌"
        
        results.append({
            'Feature': col,
            'Group1_mean': round(group1.mean(), 3),
            'Group2_mean': round(group2.mean(), 3),
            'T-Statistic': round(t_stat, 3),
            'p-value': round(p_val, 5),
            'Significant': significance
        })

    # Create DataFrame
    t_test_df = pd.DataFrame(results).sort_values(by='p-value')
    selected_features_t_test = t_test_df[t_test_df['p-value'] < 0.05]['Feature'].tolist()
    return selected_features_t_test

def tenure_category(years):
    if years < 3:
        return 'Short-term'
    elif 3 <= years < 15:
        return 'Medium-term'
    else:
        return 'Long-term'



# 2️⃣ Salary Bands
def salary_band(income):
    if income < 4755:
        return 'Low'
    elif 4755 <= income < 10329:
        return 'Medium'
    else:
        return 'High'
    


def selected_features(data):
    chi_square_features = Chi_squared_test(data)
    t_test_features = t-test_feature_selection(data)
    selected_features = chi_square_features + t_test_features
    df_selected = df[selected_features + ['attrition']]
    df_selected['tenure_category'] = df_selected['years_at_company'].apply(tenure_category)
    df_selected['salary_band'] = df_selected['monthly_income'].apply(salary_band)
    return df_selected

def label_encode(data):
    label_encode_cols = [
    'work_life_balance', 'job_satisfaction', 'overtime', 'job_level',
    'company_size', 'leadership_opportunities', 'company_reputation',
    'employee_recognition', 'age_groups', 'education_level',
    'attrition', 'tenure_category', 'salary_band'
    ]

    onehot_encode_cols = ['job_role', 'marital_status']

    # --- 2️⃣ Apply Label Encoding ---
    le = LabelEncoder()

    # Apply label encoding to each appropriate column
    for col in label_encode_cols:
        if col in df_selected.columns:
            df_selected[col] = le.fit_transform(df_selected[col].astype(str))

    # --- 3️⃣ Apply One-Hot Encoding ---
    df_encoded = pd.get_dummies(df_selected, columns=onehot_encode_cols, drop_first=True)
    df_encoded = df_encoded.astype(int)
    return df_encoded


In [33]:
# Your existing helper functions
def rows_selection(data):
    return data[~((data["years_at_company"] == 0) & (data.age > 45))]

def tenure_category(years):
    if years < 3:
        return 'Short-term'
    elif 3 <= years < 15:
        return 'Medium-term'
    else:
        return 'Long-term'

def salary_band(income):
    if income < 4755:
        return 'Low'
    elif 4755 <= income < 10329:
        return 'Medium'
    else:
        return 'High'

# === Chi-square & t-test ===
def Chi_squared_test(data):
    df = rows_selection(data)
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns
    target = 'attrition'

    results = []
    for col in categorical_cols:
        if col != target:
            contingency_table = pd.crosstab(df[col], df[target])
            chi2, p, dof, expected = chi2_contingency(contingency_table)
            if p < 0.05:
                results.append(col)
    return results

def t_test_feature_selection(data):
    df = rows_selection(data)
    target = 'attrition'
    numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
    groups = df[target].unique()
    if len(groups) != 2:
        return []
    results = []
    for col in numeric_cols:
        g1 = df[df[target] == groups[0]][col].dropna()
        g2 = df[df[target] == groups[1]][col].dropna()
        _, p = ttest_ind(g1, g2, equal_var=False)
        if p < 0.05:
            results.append(col)
    return results

# === Custom Transformers ===
# class FeatureSelector(BaseEstimator, TransformerMixin):
#     def fit(self, X, y=None):
#         chi_features = Chi_squared_test(X)
#         ttest_features = t_test_feature_selection(X)
#         self.selected_features_ = chi_features + ttest_features
#         return self

#     def transform(self, X):
#         df = X.copy()
#         df = df[self.selected_features_ + ['attrition']]
#         df['tenure_category'] = df['years_at_company'].apply(tenure_category)
#         df['salary_band'] = df['monthly_income'].apply(salary_band)
#         return df


# class LabelOneHotEncoder(BaseEstimator, TransformerMixin):
#     def fit(self, X, y=None):
#         self.label_encode_cols = [
#             'work_life_balance', 'job_satisfaction', 'overtime', 'job_level',
#             'company_size', 'leadership_opportunities', 'company_reputation',
#             'employee_recognition', 'age_groups', 'education_level',
#             'attrition', 'tenure_category', 'salary_band'
#         ]
#         self.onehot_encode_cols = ['job_role', 'marital_status']
#         self.encoders_ = {col: LabelEncoder() for col in self.label_encode_cols if col in X.columns}
#         for col, le in self.encoders_.items():
#             le.fit(X[col].astype(str))
#         return self

#     def transform(self, X):
#         df = X.copy()
#         for col, le in self.encoders_.items():
#             df[col] = le.transform(df[col].astype(str))
#         df = pd.get_dummies(df, columns=self.onehot_encode_cols, drop_first=True)
#         df = df.astype(int)
#         return df

class FeatureSelector(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        # Combine X and y for feature selection logic
        if y is None:
            raise ValueError("y (target) must be provided during fit().")
        
        df = X.copy()
        df['attrition'] = y  # temporarily attach target

        chi_features = Chi_squared_test(df)
        ttest_features = t_test_feature_selection(df)
        self.selected_features_ = chi_features + ttest_features
        return self

    def transform(self, X):
        df = X.copy()
        # Only select the features found during fit
        df = df[self.selected_features_]
        # Add derived categorical columns
        if 'years_at_company' in df.columns:
            df['tenure_category'] = df['years_at_company'].apply(tenure_category)
        if 'monthly_income' in df.columns:
            df['salary_band'] = df['monthly_income'].apply(salary_band)
        return df

In [34]:
df_train = pd.read_csv("../../data/Faker_Data/train.csv")
df_test = pd.read_csv("../../data/Faker_Data/test.csv")

In [35]:
X_train = df_train.drop('attrition', axis=1)
y_train = df_train['attrition']
X_test = df_test.drop('attrition', axis=1)
y_test = df_test['attrition']

In [36]:
standard_scale_cols = ["age"]
minmax_scale_cols = ["years_at_company", "monthly_income", "number_of_promotions"]
models = {
    "RandomForest": RandomForestClassifier(random_state=42),
    "SVM": SVC(),
    "KNN": KNeighborsClassifier(),
}

In [ ]:
# === Pipeline ===
standard_transformer = Pipeline([("standard_scaler", StandardScaler())])

minmax_transformer = Pipeline([("minmax_scaler", MinMaxScaler())])

preprocessor = ColumnTransformer(
    transformers=[
        ("standard", standard_transformer, standard_scale_cols),
        ("minmax", minmax_transformer, minmax_scale_cols),
    ],
    remainder="passthrough",
)

# --- your custom feature selection and encoding steps (from before) ---
pre_base = Pipeline(
    [("feature_selection", FeatureSelector()), ("encoding", LabelOneHotEncoder())]
)

# --- final full preprocessing pipeline ---
preprocessing_pipeline = Pipeline(
    [("feature_selection_encoding", pre_base), ("scaling", preprocessor)]
)

In [ ]:
# --- Train and evaluate each model ---
for name, clf in models.items():
    model = Pipeline(steps=[
        ("preprocessing", preprocessing_pipeline),
        ("classifier", clf)
    ])
    
    # Fit on training data
    model.fit(X_train, y_train)
    
    # Predict on test data
    preds = model.predict(X_test)
    
    # Evaluate accuracy
    acc = accuracy_score(y_test, preds)
    print(f"{name} accuracy: {acc:.3f}")


RandomForest accuracy: 0.889
SVM accuracy: 0.882
KNN accuracy: 0.869
